# Event No-Show Prediction — Outdoor English Events for Kids

**Domain:** an English e-learning platform for kids. The user account *is* the kid. Events are outdoor activities (storytelling in the park, treasure hunts, nature walks, sports days, etc.).

**Business goal:** increase event attendance / reduce no-show rates.

**ML goal:** binary classifier — given a registration, predict whether the kid will actually attend.

- Target: `attended` (1 = ATTENDED, 0 = NO_SHOW)
- Class balance: ~50/50 (balanced at generation time, so no SMOTE/undersampling needed)
- Models: **RandomForest** and **XGBoost**
- Primary metric: **ROC-AUC**; secondary: **PR-AUC**, **F1 on no-show**

Every feature here is derivable from `Event` + `EventRegistration` + a small holidays calendar — **no external APIs required**.

Pipeline: **load → clean → EDA → preprocess → train → evaluate → tune threshold → save artifacts.**

## 1. Setup

In [ ]:
# Install dependencies (uncomment on first run)
# %pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

## 2. Load raw data

In [ ]:
df_raw = pd.read_csv('dataset.csv')
print('Raw shape :', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## 3. Data cleaning

The raw export has the kinds of issues you'd expect from a real production database:

1. **Duplicate rows** — same registration written more than once.
2. **Multiple missing-value markers** — `""`, `"N/A"`, `"null"`, `"?"`, `"NA"`, `"unknown"`, plus our own sentinel `-1` for users with no history.
3. **Inconsistent text** — `event_category` sometimes lower-case, capitalized, or padded with whitespace.
4. **Wrong types** — `user_has_phone` sometimes stored as `"yes"` / `"no"` instead of `1` / `0`.
5. **Outliers** — `user_age = 99`, `event_price_tnd = 999999`, etc.

Each step prints a before/after count so the cleaning is auditable.

In [ ]:
df = df_raw.copy()
n0 = len(df)

# ---- 3.1 Duplicates ----
dup_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f'[duplicates]  removed {dup_count} rows  ->  {len(df)} remaining')

In [ ]:
# ---- 3.2 Normalize missing-value markers to NaN ----
MISSING_MARKERS = ['', 'N/A', 'null', '?', 'NA', 'unknown']
df = df.replace(MISSING_MARKERS, np.nan)

# Our own sentinel: -1 means 'no history' for these two cols
for col in ['user_past_attendance_rate', 'user_avg_rating_given']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].replace(-1, np.nan)

missing_summary = df.isna().sum()
print('Columns with missing values after sentinel handling:')
print(missing_summary[missing_summary > 0].sort_values(ascending=False))

In [ ]:
# ---- 3.3 Normalize inconsistent text in event_category ----
VALID_CATEGORIES = [
    'STORYTELLING_PARK','NATURE_VOCAB_WALK','TREASURE_HUNT','ROLE_PLAY_GAMES',
    'ENGLISH_PICNIC','SPORTS_AND_ENGLISH','OUTDOOR_THEATER','ARTS_AND_CRAFTS',
    'FIELD_TRIP','ENGLISH_CAMP_DAY',
]
before = df['event_category'].dropna().unique()
df['event_category'] = df['event_category'].astype(str).str.strip().str.upper()
df.loc[~df['event_category'].isin(VALID_CATEGORIES), 'event_category'] = np.nan
after = df['event_category'].dropna().unique()
print(f'[event_category] unique values: before={len(before)}  ->  after={len(after)}')
print('Cleaned values:', sorted(after))

In [ ]:
# ---- 3.4 Coerce wrong types ----
df['user_has_phone'] = (
    df['user_has_phone']
    .replace({'yes': 1, 'no': 0, 'Yes': 1, 'No': 0, 'YES': 1, 'NO': 0})
)
df['user_has_phone'] = pd.to_numeric(df['user_has_phone'], errors='coerce')

NUMERIC_COLS = [
    'user_id','user_age','user_account_age_days','user_total_registrations',
    'user_past_attendance_rate','user_past_no_shows','user_past_cancellations',
    'user_avg_rating_given','user_profile_completeness','user_has_phone',
    'event_id','event_max_attendees','event_current_attendees','event_capacity_utilization',
    'event_is_featured','event_is_public','event_is_free','event_price_tnd',
    'event_duration_hours','event_day_of_week','event_hour_of_day','event_is_weekend',
    'event_host_total_events','event_host_past_avg_attendance_rate',
    'days_until_event','registration_hour','sms_reminder_sent','email_reminder_sent',
    'registered_via_waitlist','is_holiday','attended',
]
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print('Dtypes after coercion (numeric only):')
print(df[NUMERIC_COLS].dtypes.value_counts())

In [ ]:
# ---- 3.5 Outlier handling (set out-of-range values to NaN, imputed later) ----
RANGES = {
    'user_age':         (4, 16),     # the platform is for kids
    'event_price_tnd':  (0, 500),
    'event_duration_hours': (0.5, 12),
}
for col, (lo, hi) in RANGES.items():
    bad = ((df[col] < lo) | (df[col] > hi)).sum()
    df.loc[df[col] < lo, col] = np.nan
    df.loc[df[col] > hi, col] = np.nan
    print(f'[outliers] {col:22s}: {bad} out-of-range values set to NaN')

In [ ]:
# ---- 3.6 Final missing-value handling ----
# `was_missing` flag for history cols (missingness signals 'new user on platform')
for col in ['user_past_attendance_rate', 'user_avg_rating_given']:
    df[col + '_missing'] = df[col].isna().astype(int)

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median(numeric_only=True))
df[cat_cols] = df[cat_cols].fillna('UNKNOWN')

df = df[df['attended'].isin([0, 1])].reset_index(drop=True)
df['attended'] = df['attended'].astype(int)

print(f'[done] cleaned shape: {df.shape}  (started at {n0})')
print(f'Remaining missing values: {int(df.isna().sum().sum())}')

In [ ]:
# ---- 3.7 Feature engineering — kid's English level vs event's target level ----
# A kid registered for an event above/below their level is more likely to skip.
df['user_event_level_match'] = (
    (df['event_target_level'] == 'ALL_LEVELS') |
    (df['user_english_level'] == df['event_target_level'])
).astype(int)
print(df['user_event_level_match'].value_counts(normalize=True).round(3))

## 4. EDA on the cleaned data

In [ ]:
balance = df['attended'].value_counts(normalize=True).rename({1: 'attended', 0: 'no_show'})
print(balance)
balance.plot(kind='bar', title='Class balance (cleaned)', color=['#2ecc71', '#e74c3c'])
plt.ylabel('Proportion'); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
df.groupby('event_category')['attended'].mean().sort_values().plot(
    kind='barh', ax=axes[0,0], title='Attendance rate by event category')
df.groupby('sms_reminder_sent')['attended'].mean().plot(
    kind='bar', ax=axes[0,1], title='Attendance rate vs SMS reminder')
sns.regplot(x='days_until_event', y='attended', data=df, ax=axes[1,0],
            logistic=True, scatter_kws={'alpha': 0.05}, line_kws={'color': 'red'})
axes[1,0].set_title('Attendance vs lead time')
sns.regplot(x='user_past_attendance_rate', y='attended', data=df, ax=axes[1,1],
            logistic=True, scatter_kws={'alpha': 0.05}, line_kws={'color': 'red'})
axes[1,1].set_title('Attendance vs user past attendance rate')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.groupby('user_age')['attended'].mean().plot(kind='bar', ax=axes[0], title='Attendance by user age (kid)')
df.groupby('user_event_level_match')['attended'].mean().plot(
    kind='bar', ax=axes[1], title='Level match -> attendance', color='#9b59b6')
plt.tight_layout(); plt.show()

In [ ]:
corr_cols = [
    'user_past_attendance_rate', 'user_past_no_shows', 'user_total_registrations',
    'days_until_event', 'sms_reminder_sent', 'email_reminder_sent',
    'event_is_featured', 'event_is_free', 'event_host_past_avg_attendance_rate',
    'user_age', 'user_event_level_match', 'event_is_weekend', 'is_holiday', 'attended'
]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation with attendance'); plt.show()

## 5. Preprocessing for modeling

- Drop ID columns (`user_id`, `event_id`) — identity, not signal.
- One-hot encode categoricals.
- Numericals are passed through (tree models don't need scaling).
- Stratified train/test split (80/20).

In [ ]:
TARGET = 'attended'
DROP_COLS = ['user_id', 'event_id']
CAT_COLS = ['event_category', 'event_target_level', 'event_skill_focus',
            'user_gender', 'user_english_level']
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS + [TARGET]]
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

X = df[FEATURE_COLS]
y = df[TARGET]
print('numeric features :', len(NUM_COLS))
print('cat     features :', len(CAT_COLS))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('train:', X_train.shape, ' test:', X_test.shape)
print('train balance:', y_train.value_counts(normalize=True).round(3).to_dict())

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
        ('num', 'passthrough', NUM_COLS),
    ]
)

## 6. Model 1 — RandomForest

In [ ]:
rf_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

rf_pipe.fit(X_train, y_train)
rf_proba = rf_pipe.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

print(f'ROC-AUC : {roc_auc_score(y_test, rf_proba):.4f}')
print(f'PR-AUC  : {average_precision_score(y_test, rf_proba):.4f}')
print(classification_report(y_test, rf_pred, target_names=['no_show', 'attended']))

## 7. Model 2 — XGBoost

In [ ]:
xgb_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        eval_metric='auc',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_pipe.fit(X_train, y_train)
xgb_proba = xgb_pipe.predict_proba(X_test)[:, 1]
xgb_pred  = (xgb_proba >= 0.5).astype(int)

print(f'ROC-AUC : {roc_auc_score(y_test, xgb_proba):.4f}')
print(f'PR-AUC  : {average_precision_score(y_test, xgb_proba):.4f}')
print(classification_report(y_test, xgb_pred, target_names=['no_show', 'attended']))

## 8. Compare the two models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, proba in [('RandomForest', rf_proba), ('XGBoost', xgb_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, proba):.3f})')
    pr, rc, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rc, pr, label=f'{name} (AP={average_precision_score(y_test, proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Precision-Recall'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pred) in zip(axes, [('RandomForest', rf_pred), ('XGBoost', xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['no_show', 'attended'], yticklabels=['no_show', 'attended'])
    ax.set_title(f'{name} (threshold=0.5)'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

## 9. Threshold tuning — optimize for catching no-shows

In [ ]:
def best_threshold_for_no_show(y_true, proba):
    no_show_proba = 1 - proba
    y_no_show = (y_true == 0).astype(int)
    thresholds = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_no_show, (no_show_proba >= t).astype(int)) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return thresholds[best_idx], f1s[best_idx]

for name, proba in [('RandomForest', rf_proba), ('XGBoost', xgb_proba)]:
    t, f1 = best_threshold_for_no_show(y_test, proba)
    print(f'{name:14s}  best no-show threshold (on 1 - p_attend) = {t:.2f}  F1(no_show) = {f1:.3f}')

## 10. Feature importance

In [ ]:
def importances_df(pipe, top_n=20):
    feature_names = pipe.named_steps['prep'].get_feature_names_out()
    imp = pipe.named_steps['clf'].feature_importances_
    return pd.Series(imp, index=feature_names).sort_values(ascending=False).head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
importances_df(rf_pipe).iloc[::-1].plot(kind='barh', ax=axes[0], title='RandomForest — top features')
importances_df(xgb_pipe).iloc[::-1].plot(kind='barh', ax=axes[1], title='XGBoost — top features', color='#e67e22')
plt.tight_layout(); plt.show()

## 11. Cross-validated AUC (overfitting sanity check)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_auc  = cross_val_score(rf_pipe,  X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
xgb_auc = cross_val_score(xgb_pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'RandomForest 5-fold ROC-AUC : {rf_auc.mean():.4f} ± {rf_auc.std():.4f}')
print(f'XGBoost      5-fold ROC-AUC : {xgb_auc.mean():.4f} ± {xgb_auc.std():.4f}')

## 12. Save artifacts

In [ ]:
joblib.dump(rf_pipe,  'rf_pipeline.joblib')
joblib.dump(xgb_pipe, 'xgb_pipeline.joblib')
print('Saved: rf_pipeline.joblib, xgb_pipeline.joblib')

## 13. Inference example

In [ ]:
model = joblib.load('xgb_pipeline.joblib')
sample = X_test.iloc[[0]].copy()
p_attend = model.predict_proba(sample)[0, 1]
p_no_show = 1 - p_attend
print(f'P(attend)  = {p_attend:.3f}')
print(f'P(no_show) = {p_no_show:.3f}')
print(f'Recommended action: {"send extra SMS reminder" if p_no_show > 0.4 else "standard flow"}')

## Cleaning summary (talk track for the mentor)

1. **Removed duplicate rows** — ~2% of the export was duplicated.
2. **Standardized missing markers** — `""`, `"N/A"`, `"null"`, `"?"`, `"NA"`, `"unknown"`, and the `-1` sentinel all became `NaN`.
3. **Normalized text** — stripped whitespace and upper-cased `event_category`, validated against the known enum.
4. **Coerced types** — `user_has_phone` had `"yes"`/`"no"` strings; converted to numeric.
5. **Out-of-range outliers** — `user_age=99`, `event_price_tnd=999999`, etc. set to `NaN` against documented domain ranges.
6. **Imputation** — `was_missing` flag for the user-history columns (the missingness itself signals a new user), then median for numerics and `'UNKNOWN'` bucket for categoricals.
7. **Feature engineering** — added `user_event_level_match` (kid's English level matches the event's target level).
8. **Class balance** — already 50/50 at generation time, so no SMOTE/undersampling needed.

**Why no weather / distance / sibling features:** every feature in this notebook is computable from your existing Spring entities (`Event`, `EventRegistration`) plus a Tunisian holidays list — no third-party APIs, no extra schema changes.